# Ejercicio: analizar_texto con mocks

In [14]:
import sys
!{sys.executable} -m pip install requests

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached certifi-2026.4.22-py3-none-any.whl (135 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ------------------------ --------------- 3/5 [certifi]
   -------------------------------- ------- 4/5 [requests]
   ---------------------------------------- 5/5 [requests]




[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import sys
import os

# Sube un nivel desde notebooks/ hasta la raíz del proyecto
sys.path.insert(0, os.path.abspath('..'))

from unittest.mock import patch, Mock
from src.analizador import analizar_texto
import requests

## Caso éxito

In [21]:
def test_exito():
    mock_resp = Mock()
    mock_resp.text = 'hola\nmundo'
    mock_resp.raise_for_status = Mock()
    with patch('src.analizador.requests.get', return_value=mock_resp) as mock_get:
        lineas, chars = analizar_texto('fake')
        assert lineas == 2
        assert chars == 9
        mock_get.assert_called_once()

# Fallo y luego exito


In [22]:
def test_fallo_y_reintento():
    mock_fallo = Mock()
    mock_fallo.raise_for_status = Mock(side_effect=requests.RequestException("Error HTTP"))
    # side_effect hace que al llamar raise_for_status() se lance la excepción

    mock_exito = Mock()
    mock_exito.text = 'hola\nmundo'
    mock_exito.raise_for_status = Mock()

    # side_effect como lista: primera llamada retorna fallo, segunda retorna éxito
    with patch('src.analizador.requests.get', side_effect=[mock_fallo, mock_exito]) as mock_get:
        lineas, chars = analizar_texto('http://fake.url')

        assert lineas == 2
        assert chars == 9
        assert mock_get.call_count == 2   # tuvo que intentar dos veces

test_fallo_y_reintento()
print("test_fallo_y_reintento pasó")

test_fallo_y_reintento pasó


# Fallo total

In [23]:
def test_fallo_total():
    mock_fallo = Mock()
    mock_fallo.raise_for_status = Mock(side_effect=requests.RequestException("Error HTTP"))

    with patch('src.analizador.requests.get', return_value=mock_fallo) as mock_get:
        try:
            analizar_texto('http://fake.url')
            assert False, "Debería haber lanzado RuntimeError"
        except RuntimeError as e:
            assert "3 intentos" in str(e)
        
        assert mock_get.call_count == 3

test_fallo_total()
print("test_fallo_total pasó")

test_fallo_total pasó


## Ejercicio
1. Simular fallo y reintento
2. Simular fallo total (3 intentos)